In [1]:
# Cell 41: Check if MmCows fold definitions exist
!echo "=== Any fold files? ==="
!find /home/oussema/pfa-mmcows/ -type f -name "*fold*" 2>/dev/null | head -10
!echo "\n=== Check notebooks for fold info ==="
!find /home/oussema/pfa-mmcows/ -type f -name "*split*" 2>/dev/null | head -10

=== Any fold files? ===
\n=== Check notebooks for fold info ===


In [2]:
# Cell 42: Build 5-fold cross-validation index
#
# 5-fold means we split the data into 5 equal chunks.
# Each time, 3 chunks for training (60%), 1 for validation (20%), 1 for test (20%).
# We rotate which chunk is val/test — so every image gets tested exactly once.
#
# We split by TIMESTAMP (same as before) so all 4 cameras at same moment
# go to the same fold — no data leakage.

import pandas as pd
import numpy as np
from pathlib import Path

IMAGES = Path("/home/oussema/pfa-mmcows/store/data/raw/visual_data/images/0725")
LABELS = Path("/home/oussema/pfa-mmcows/store/data/raw/visual_data/labels/combined/0725")
CAMERAS = ["cam_1", "cam_2", "cam_3", "cam_4"]

# Build master index
rows = []
for cam in CAMERAS:
    for img_path in sorted((IMAGES / cam).glob("*.jpg")):
        lbl_path = LABELS / cam / (img_path.stem + ".txt")
        rows.append({
            "camera": cam,
            "timestamp": img_path.stem.split("_")[0],
            "image_path": str(img_path),
            "label_path": str(lbl_path),
            "has_label": lbl_path.exists()
        })

df = pd.DataFrame(rows)

# Sort unique timestamps and split into 5 equal groups
unique_ts = sorted(df["timestamp"].unique())
n = len(unique_ts)
fold_size = n // 5

# Assign each timestamp to a fold (0-4)
ts_to_fold = {}
for i, ts in enumerate(unique_ts):
    fold = min(i // fold_size, 4)  # last fold absorbs remainder
    ts_to_fold[ts] = fold

df["fold"] = df["timestamp"].map(ts_to_fold)

# Define 5-fold splits: each fold rotates which chunk is train/val/test
# Fold 0: test=chunk0, val=chunk1, train=chunk2,3,4
# Fold 1: test=chunk1, val=chunk2, train=chunk3,4,0
# etc.
fold_assignments = {
    0: {"test": [0], "val": [1], "train": [2, 3, 4]},
    1: {"test": [1], "val": [2], "train": [3, 4, 0]},
    2: {"test": [2], "val": [3], "train": [4, 0, 1]},
    3: {"test": [3], "val": [4], "train": [0, 1, 2]},
    4: {"test": [4], "val": [0], "train": [1, 2, 3]},
}

# Show distribution
print(f"Total samples: {len(df)}")
print(f"Unique timestamps: {len(unique_ts)}")
print(f"\nSamples per chunk:")
print(df["fold"].value_counts().sort_index())

# Save
out_path = Path("/home/oussema/pfa-mmcows/store/data/clean/dataset_index_5fold.csv")
df.to_csv(out_path, index=False)
print(f"\n✅ Saved to {out_path}")

# Also save fold assignments for reference
import json
json_path = Path("/home/oussema/pfa-mmcows/store/data/clean/fold_assignments.json")
json_path.write_text(json.dumps(fold_assignments, indent=2))
print(f"✅ Fold assignments saved to {json_path}")

Total samples: 20153
Unique timestamps: 5040

Samples per chunk:
fold
0    4028
1    4029
2    4032
3    4032
4    4032
Name: count, dtype: int64

✅ Saved to /home/oussema/pfa-mmcows/store/data/clean/dataset_index_5fold.csv
✅ Fold assignments saved to /home/oussema/pfa-mmcows/store/data/clean/fold_assignments.json


In [ ]:
# Cell 43: Build YOLO 16-class dataset for all 5 folds
#
# For each fold (0-4), we create:
#   yolo_16class/fold_X/images/train/  val/  test/
#   yolo_16class/fold_X/labels/train/  val/  test/
#   yolo_16class/fold_X/dataset.yaml
#
# Labels stay as-is (cow_id 0-15 = 16 classes)
# Images are symlinked

import pandas as pd
import json
from pathlib import Path

df = pd.read_csv("/home/oussema/pfa-mmcows/store/data/clean/dataset_index_5fold.csv")
fold_assignments = json.loads(Path("/home/oussema/pfa-mmcows/store/data/clean/fold_assignments.json").read_text())
YOLO_BASE = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_16class")

COW_NAMES = [f"cow_{i}" for i in range(1,17)]

for fold_idx in range(5):
    assignment = fold_assignments[str(fold_idx)]
    fold_dir = YOLO_BASE / f"fold_{fold_idx}"
    
    for split in ["train", "val", "test"]:
        (fold_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (fold_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
    
    count = {"train": 0, "val": 0, "test": 0}
    
    for _, row in df.iterrows():
        chunk = row["fold"]
        cam = row["camera"]
        fname = Path(row["image_path"]).stem
        
        # Determine split for this chunk in this fold
        split = None
        for s in ["train", "val", "test"]:
            if chunk in assignment[s]:
                split = s
                break
        
        # Symlink image
        img_src = Path(row["image_path"])
        img_dst = fold_dir / "images" / split / f"{cam}_{fname}.jpg"
        if not img_dst.exists():
            img_dst.symlink_to(img_src)
        
        # Symlink original label (16 classes, no remapping)
        lbl_src = Path(row["label_path"])
        lbl_dst = fold_dir / "labels" / split / f"{cam}_{fname}.txt"
        if lbl_src.exists() and not lbl_dst.exists():
            lbl_dst.symlink_to(lbl_src)
        
        count[split] += 1
    
    # Write YAML
    yaml_content = f"""path: {fold_dir}
train: images/train
val: images/val
test: images/test

nc: 16
names: {COW_NAMES}
"""
    (fold_dir / "dataset.yaml").write_text(yaml_content)
    
    print(f"Fold {fold_idx}: train={count['train']}, val={count['val']}, test={count['test']}")

print(f"\n✅ All 5 folds built under {YOLO_BASE}")

Fold 0: train=12096, val=4029, test=4028
Fold 1: train=12092, val=4032, test=4029
Fold 2: train=12089, val=4032, test=4032
Fold 3: train=12089, val=4032, test=4032
Fold 4: train=12093, val=4028, test=4032

✅ All 5 folds built under /home/oussema/pfa-mmcows/store/data/clean/yolo_16class


In [1]:
# Cell 43b: Rebuild 5-fold index using MmCows official fold config
#
# MmCows splits the day into 2 groups of 5 time periods each.
# Group 1: early morning + evening (2:57-6:00, 18:00-23:57)
# Group 2: daytime (6:00-18:00)
# Each fold rotates which periods are train(60%)/val(20%)/test(20%)
#
# We assign each image to a period based on its timestamp,
# then map period → fold split using the official config.

import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

df = pd.read_csv("/home/oussema/pfa-mmcows/store/data/clean/dataset_index_5fold.csv")

# Convert timestamp to datetime for time-range matching
df["datetime"] = pd.to_datetime(df["timestamp"].astype(float), unit="s", utc=True)

# Define the 10 time periods (5 per group)
# Each period: (period_id, start_datetime, end_datetime)
time_periods = {
    "1": [
        ("2023-07-25 02:57:18", "2023-07-25 04:30:00"),  # group_1
        ("2023-07-25 06:00:00", "2023-07-25 08:24:00"),  # group_2
    ],
    "2": [
        ("2023-07-25 04:30:00", "2023-07-25 06:00:00"),
        ("2023-07-25 08:24:00", "2023-07-25 10:48:00"),
    ],
    "3": [
        ("2023-07-25 18:00:00", "2023-07-25 20:00:00"),
        ("2023-07-25 10:48:00", "2023-07-25 13:12:00"),
    ],
    "4": [
        ("2023-07-25 20:00:00", "2023-07-25 22:00:00"),
        ("2023-07-25 13:12:00", "2023-07-25 15:36:00"),
    ],
    "5": [
        ("2023-07-25 22:00:00", "2023-07-25 23:57:16"),
        ("2023-07-25 15:36:00", "2023-07-25 18:00:00"),
    ],
}

# Fold definitions (official MmCows)
folds_config = {
    0: {"train": ["1","2","3"], "val": ["4"], "test": ["5"]},
    1: {"train": ["2","3","4"], "val": ["5"], "test": ["1"]},
    2: {"train": ["3","4","5"], "val": ["1"], "test": ["2"]},
    3: {"train": ["4","5","1"], "val": ["2"], "test": ["3"]},
    4: {"train": ["5","1","2"], "val": ["3"], "test": ["4"]},
}

# Assign each image to a period
def assign_period(dt):
    """Find which period (1-5) this datetime belongs to."""
    dt_str = dt.strftime("%Y-%m-%d %H:%M:%S")
    for period_id, ranges in time_periods.items():
        for start, end in ranges:
            if start <= dt_str < end:
                return period_id
    return None

df["period"] = df["datetime"].apply(assign_period)

# Check how many images got assigned
assigned = df["period"].notna().sum()
total = len(df)
print(f"Assigned: {assigned}/{total} ({100*assigned/total:.1f}%)")
print(f"\nSamples per period:")
print(df["period"].value_counts().sort_index())

# Assign fold splits for each fold
for fold_idx in range(5):
    col = f"fold_{fold_idx}"
    config = folds_config[fold_idx]
    def get_split(period):
        if period is None: return None
        if period in config["train"]: return "train"
        if period in config["val"]: return "val"
        if period in config["test"]: return "test"
        return None
    df[col] = df["period"].apply(get_split)

# Save
out_path = Path("/home/oussema/pfa-mmcows/store/data/clean/dataset_index_5fold_official.csv")
df.to_csv(out_path, index=False)

# Show fold 0 distribution
print(f"\nFold 0 split distribution:")
print(df["fold_0"].value_counts())

print(f"\n✅ Saved to {out_path}")

Assigned: 15353/20153 (76.2%)

Samples per period:
period
1     428
2    2304
3    4217
4    4224
5    4180
Name: count, dtype: int64

Fold 0 split distribution:
fold_0
train    6949
val      4224
test     4180
Name: count, dtype: int64

✅ Saved to /home/oussema/pfa-mmcows/store/data/clean/dataset_index_5fold_official.csv


In [2]:
# Cell 43c: Debug unassigned images
#
# Check which timestamps are falling outside the time ranges

unassigned = df[df["period"].isna()]
print(f"Unassigned: {len(unassigned)}")
print(f"\nTime range of unassigned:")
print(f"  Min: {unassigned['datetime'].min()}")
print(f"  Max: {unassigned['datetime'].max()}")

print(f"\nSample unassigned timestamps:")
print(unassigned["datetime"].head(10).tolist())

print(f"\nFull dataset time range:")
print(f"  Min: {df['datetime'].min()}")
print(f"  Max: {df['datetime'].max()}")

Unassigned: 4800

Time range of unassigned:
  Min: 2023-07-25 23:57:26+00:00
  Max: 2023-07-26 04:57:11+00:00

Sample unassigned timestamps:
[Timestamp('2023-07-25 23:57:26+0000', tz='UTC'), Timestamp('2023-07-25 23:57:41+0000', tz='UTC'), Timestamp('2023-07-25 23:57:56+0000', tz='UTC'), Timestamp('2023-07-25 23:58:11+0000', tz='UTC'), Timestamp('2023-07-25 23:58:26+0000', tz='UTC'), Timestamp('2023-07-25 23:58:41+0000', tz='UTC'), Timestamp('2023-07-25 23:58:56+0000', tz='UTC'), Timestamp('2023-07-25 23:59:11+0000', tz='UTC'), Timestamp('2023-07-25 23:59:26+0000', tz='UTC'), Timestamp('2023-07-25 23:59:41+0000', tz='UTC')]

Full dataset time range:
  Min: 2023-07-25 07:57:26+00:00
  Max: 2023-07-26 04:57:11+00:00


In [3]:
# Cell 43d: Fix timezone — convert UTC timestamps to CDT (UTC-5)
#
# MmCows fold config uses CDT (Central Daylight Time = UTC-5)
# Our timestamps are in UTC. We need to shift by -5 hours
# so the time ranges match.

import pandas as pd
from pathlib import Path
from datetime import timedelta

df = pd.read_csv("/home/oussema/pfa-mmcows/store/data/clean/dataset_index_5fold.csv")
df["datetime"] = pd.to_datetime(df["timestamp"].astype(float), unit="s", utc=True)

# Convert to CDT (UTC-5)
df["datetime_cdt"] = df["datetime"] - timedelta(hours=5)
df["datetime_cdt_str"] = df["datetime_cdt"].dt.strftime("%Y-%m-%d %H:%M:%S")

print(f"CDT time range:")
print(f"  Min: {df['datetime_cdt'].min()}")
print(f"  Max: {df['datetime_cdt'].max()}")

# Same time_periods as before
time_periods = {
    "1": [
        ("2023-07-25 02:57:18", "2023-07-25 04:30:00"),
        ("2023-07-25 06:00:00", "2023-07-25 08:24:00"),
    ],
    "2": [
        ("2023-07-25 04:30:00", "2023-07-25 06:00:00"),
        ("2023-07-25 08:24:00", "2023-07-25 10:48:00"),
    ],
    "3": [
        ("2023-07-25 18:00:00", "2023-07-25 20:00:00"),
        ("2023-07-25 10:48:00", "2023-07-25 13:12:00"),
    ],
    "4": [
        ("2023-07-25 20:00:00", "2023-07-25 22:00:00"),
        ("2023-07-25 13:12:00", "2023-07-25 15:36:00"),
    ],
    "5": [
        ("2023-07-25 22:00:00", "2023-07-25 23:57:16"),
        ("2023-07-25 15:36:00", "2023-07-25 18:00:00"),
    ],
}

folds_config = {
    0: {"train": ["1","2","3"], "val": ["4"], "test": ["5"]},
    1: {"train": ["2","3","4"], "val": ["5"], "test": ["1"]},
    2: {"train": ["3","4","5"], "val": ["1"], "test": ["2"]},
    3: {"train": ["4","5","1"], "val": ["2"], "test": ["3"]},
    4: {"train": ["5","1","2"], "val": ["3"], "test": ["4"]},
}

def assign_period(dt_cdt_str):
    for period_id, ranges in time_periods.items():
        for start, end in ranges:
            if start <= dt_cdt_str < end:
                return period_id
    return None

df["period"] = df["datetime_cdt_str"].apply(assign_period)

assigned = df["period"].notna().sum()
print(f"\nAssigned: {assigned}/{len(df)} ({100*assigned/len(df):.1f}%)")
print(f"\nSamples per period:")
print(df["period"].value_counts().sort_index())

# Assign fold splits
for fold_idx in range(5):
    col = f"fold_{fold_idx}"
    config = folds_config[fold_idx]
    def get_split(period, cfg=config):
        if period is None: return None
        if period in cfg["train"]: return "train"
        if period in cfg["val"]: return "val"
        if period in cfg["test"]: return "test"
        return None
    df[col] = df["period"].apply(get_split)

print(f"\nFold 0 distribution:")
print(df["fold_0"].value_counts())

# Save
out_path = Path("/home/oussema/pfa-mmcows/store/data/clean/dataset_index_5fold_official.csv")
df.to_csv(out_path, index=False)
print(f"\n✅ Saved to {out_path}")

CDT time range:
  Min: 2023-07-25 02:57:26+00:00
  Max: 2023-07-25 23:57:11+00:00

Assigned: 20153/20153 (100.0%)

Samples per period:
period
1    3781
2    3744
3    4224
4    4224
5    4180
Name: count, dtype: int64

Fold 0 distribution:
fold_0
train    11749
val       4224
test      4180
Name: count, dtype: int64

✅ Saved to /home/oussema/pfa-mmcows/store/data/clean/dataset_index_5fold_official.csv


In [4]:
# Cell 44: Build YOLO 16-class dataset for all 5 folds (official MmCows splits)
#
# Same as before but using corrected fold assignments.
# Labels stay as-is (cow_id 0-15 = 16 classes).
# Symlinks for both images and labels.

import pandas as pd
from pathlib import Path
import shutil

df = pd.read_csv("/home/oussema/pfa-mmcows/store/data/clean/dataset_index_5fold_official.csv")
YOLO_BASE = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_16class")

# Remove old fold dirs if they exist
if YOLO_BASE.exists():
    shutil.rmtree(YOLO_BASE)

COW_NAMES = [f"cow_{i}" for i in range(1,17)]

for fold_idx in range(5):
    fold_dir = YOLO_BASE / f"fold_{fold_idx}"
    col = f"fold_{fold_idx}"
    
    for split in ["train", "val", "test"]:
        (fold_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (fold_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
    
    count = {"train": 0, "val": 0, "test": 0}
    
    for _, row in df.iterrows():
        split = row[col]
        if pd.isna(split):
            continue
        
        cam = row["camera"]
        fname = Path(row["image_path"]).stem
        
        # Symlink image
        img_src = Path(row["image_path"])
        img_dst = fold_dir / "images" / split / f"{cam}_{fname}.jpg"
        if not img_dst.exists():
            img_dst.symlink_to(img_src)
        
        # Symlink label (original 16-class)
        lbl_src = Path(row["label_path"])
        lbl_dst = fold_dir / "labels" / split / f"{cam}_{fname}.txt"
        if lbl_src.exists() and not lbl_dst.exists():
            lbl_dst.symlink_to(lbl_src)
        
        count[split] += 1
    
    # Write YAML
    yaml_content = f"""path: {fold_dir}
train: images/train
val: images/val
test: images/test

nc: 16
names: {COW_NAMES}
"""
    (fold_dir / "dataset.yaml").write_text(yaml_content)
    
    print(f"Fold {fold_idx}: train={count['train']}, val={count['val']}, test={count['test']}")

print(f"\n✅ All 5 folds built under {YOLO_BASE}")

Fold 0: train=11749, val=4224, test=4180
Fold 1: train=12192, val=4180, test=3781
Fold 2: train=12628, val=3781, test=3744
Fold 3: train=12185, val=3744, test=4224
Fold 4: train=11705, val=4224, test=4224

✅ All 5 folds built under /home/oussema/pfa-mmcows/store/data/clean/yolo_16class


## Training (my folds)

In [ ]:
# Cell 44: Train YOLOv8-n on all 5 folds (16 classes)
#
# We train 5 separate models — one per fold.
# Each model sees different train/val/test data.
# At the end, we average the metrics across all 5 folds
# to get a robust estimate of performance.
#
# Think of it like: 5 students take 5 different exams.
# The class average tells us how good the "teaching method" is,
# not just how lucky one student got.

from ultralytics import YOLO
from pathlib import Path

YOLO_BASE = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_16class")

for fold_idx in range(5):
    print(f"\n{'='*50}")
    print(f"TRAINING FOLD {fold_idx}")
    print(f"{'='*50}")
    
    model = YOLO("yolov8n.pt")
    
    model.train(
        data=str(YOLO_BASE / f"fold_{fold_idx}" / "dataset.yaml"),
        epochs=50,
        imgsz=640,
        batch=32,
        device=0,
        workers=8,
        amp=True,
        project="/home/oussema/pfa-mmcows/runs/detect_16class",
        name=f"fold_{fold_idx}",
        patience=10,
        save=True,
        plots=True,
    )

print("\n✅ All 5 folds trained")

## Training (mmcows folds)

In [ ]:
# Cell 44: Train YOLOv8-n on all 5 folds (16 classes)
#
# We train 5 separate models — one per fold.
# Each model sees different train/val/test data.
# At the end, we average the metrics across all 5 folds
# to get a robust estimate of performance.
#
# Think of it like: 5 students take 5 different exams.
# The class average tells us how good the "teaching method" is,
# not just how lucky one student got.

from ultralytics import YOLO
from pathlib import Path

YOLO_BASE = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_16class")

for fold_idx in range(5):
    print(f"\n{'='*50}")
    print(f"TRAINING FOLD {fold_idx}")
    print(f"{'='*50}")
    
    model = YOLO("yolov8n.pt")
    
    model.train(
        data=str(YOLO_BASE / f"fold_{fold_idx}" / "dataset.yaml"),
        epochs=50,
        imgsz=640,
        batch=32,
        device=0,
        workers=8,
        amp=True,
        project="/home/oussema/pfa-mmcows/runs/detect_16class",
        name=f"fold_{fold_idx}",
        patience=10,
        save=True,
        plots=True,
    )

print("\n✅ All 5 folds trained")


TRAINING FOLD 0
New https://pypi.org/project/ultralytics/8.4.48 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.39 🚀 Python-3.13.12 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 7706MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/oussema/pfa-mmcows/store/data/clean/yolo_16class/fold_0/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Overriding model.yaml nc=80 with nc=17

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytic

/opt/miniconda3/lib/python3.13/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


MLflow: logging run_id(a547127d3cee42eea5b7976d5a57312d) to runs/mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs/mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/oussema/pfa-mmcows/runs/detect_16class/fold_0-4
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      4.35G      2.178      4.379      1.529        501        640: 24% ━━╸───────── 90/368 1.6it/s 1:51<2:583

In [5]:
# Cell 45c: Fix YAML to nc=17 (class 0 unused, cows are 1-16)
#
# Instead of remapping labels, we just tell YOLO there are 17 classes.
# Class 0 will be empty (no cow uses it), classes 1-16 are the 16 cows.
# YOLO handles unused classes fine — it just learns nothing for class 0.

from pathlib import Path

YOLO_BASE = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_16class")
COW_NAMES = ["unused"] + [f"cow_{i}" for i in range(1, 17)]

for fold_idx in range(5):
    fold_dir = YOLO_BASE / f"fold_{fold_idx}"
    yaml_content = f"""path: {fold_dir}
train: images/train
val: images/val
test: images/test

nc: 17
names: {COW_NAMES}
"""
    (fold_dir / "dataset.yaml").write_text(yaml_content)

print("✅ All 5 YAMLs updated: nc=17, class 0=unused, classes 1-16=cows")

✅ All 5 YAMLs updated: nc=17, class 0=unused, classes 1-16=cows
